In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
from PIL import Image
import os

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

In [ ]:
# If the Kaggle CelebA dataset is attached, use it directly.
# Otherwise pull it via kagglehub (Kaggle API, not Google Drive).
# Google Drive is a last resort: torchvision's CelebA download hits a shared
# Drive file that frequently runs into Google's "too many downloads" quota
# and can stay broken for hours, regardless of retries.

KAGGLE_IMG_DIR = '/kaggle/input/celeba-dataset/img_align_celeba/img_align_celeba'
LOCAL_IMG_DIR  = 'img_align_celeba'


def find_image_dir(root):
    """Find the directory under root that directly contains the .jpg files."""
    for dirpath, _, filenames in os.walk(root):
        if any(f.endswith('.jpg') for f in filenames):
            return dirpath
    return None


if os.path.exists(KAGGLE_IMG_DIR):
    IMG_DIR = KAGGLE_IMG_DIR
elif os.path.exists(LOCAL_IMG_DIR):
    IMG_DIR = LOCAL_IMG_DIR
else:
    IMG_DIR = None
    try:
        import kagglehub
        path = kagglehub.dataset_download('jessicali9530/celeba-dataset')
        print(f'kagglehub downloaded dataset to: {path}')
        IMG_DIR = find_image_dir(path)
        print(f'Resolved image dir: {IMG_DIR}')
    except Exception as e:
        print(f'kagglehub fetch failed ({e}); falling back to Google Drive via torchvision.')

    if IMG_DIR is None or not os.path.exists(IMG_DIR):
        try:
            import gdown  # noqa: F401  (torchvision's CelebA download needs this)
        except ImportError:
            import subprocess, sys
            subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'gdown'], check=True)

        from torchvision.datasets import CelebA

        DATA_ROOT = '/kaggle/working/data' if os.path.exists('/kaggle/working') else './data'
        # split='all' + download=True grabs images + annotations from Google Drive.
        CelebA(root=DATA_ROOT, split='all', download=True)
        IMG_DIR = os.path.join(DATA_ROOT, 'celeba', 'img_align_celeba')

img_files = sorted([
    os.path.join(IMG_DIR, f)
    for f in os.listdir(IMG_DIR)
    if f.endswith('.jpg')
])

print(f'Found {len(img_files)} images')
print('Example:', img_files[0])

In [ ]:
class CelebADataset(Dataset):
    def __init__(self, img_files, img_size=64):
        self.img_files = img_files
        self.transform = T.Compose([
            T.CenterCrop(148),          # crop face region
            T.Resize((img_size, img_size)),
            T.ToTensor(),               # (C, H, W) float [0, 1]
        ])

    def __len__(self):
        return len(self.img_files)

    def __getitem__(self, idx):
        img = Image.open(self.img_files[idx]).convert('RGB')
        return self.transform(img)


dataset = CelebADataset(img_files, img_size=64)
loader  = DataLoader(dataset, batch_size=128, shuffle=True, num_workers=2, pin_memory=True)
print(f'Dataset: {len(dataset)} images  |  Batches/epoch: {len(loader)}')

# Preview
sample = torch.stack([dataset[i] for i in range(8)])
fig, axes = plt.subplots(1, 8, figsize=(16, 2.5))
for i, ax in enumerate(axes):
    ax.imshow(sample[i].permute(1, 2, 0))
    ax.axis('off')
plt.suptitle('CelebA samples (64x64)')
plt.tight_layout()
plt.show()

In [ ]:
class CelebAVAE(nn.Module):
    def __init__(self, latent_dim=128):
        super().__init__()
        self.latent_dim = latent_dim

        # Encoder: 64x64x3 -> 4x4x512 -> latent
        self.encoder = nn.Sequential(
            nn.Conv2d(3,   64,  4, 2, 1),  # 32x32
            nn.BatchNorm2d(64),
            nn.LeakyReLU(0.2),
            nn.Conv2d(64,  128, 4, 2, 1),  # 16x16
            nn.BatchNorm2d(128),
            nn.LeakyReLU(0.2),
            nn.Conv2d(128, 256, 4, 2, 1),  # 8x8
            nn.BatchNorm2d(256),
            nn.LeakyReLU(0.2),
            nn.Conv2d(256, 512, 4, 2, 1),  # 4x4
            nn.BatchNorm2d(512),
            nn.LeakyReLU(0.2),
            nn.Flatten(),                  # 512*4*4 = 8192
        )
        self.fc_mu     = nn.Linear(8192, latent_dim)
        self.fc_logvar = nn.Linear(8192, latent_dim)

        # Decoder: latent -> 4x4x512 -> 64x64x3
        self.fc_decode = nn.Linear(latent_dim, 8192)
        self.decoder = nn.Sequential(
            nn.Unflatten(1, (512, 4, 4)),
            nn.ConvTranspose2d(512, 256, 4, 2, 1),  # 8x8
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.ConvTranspose2d(256, 128, 4, 2, 1),  # 16x16
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.ConvTranspose2d(128, 64,  4, 2, 1),  # 32x32
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.ConvTranspose2d(64,  3,   4, 2, 1),  # 64x64
            nn.Sigmoid(),
        )

    def encode(self, x):
        h = self.encoder(x)
        return self.fc_mu(h), self.fc_logvar(h)

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        return mu + std * torch.randn_like(std)

    def decode(self, z):
        return self.decoder(self.fc_decode(z))

    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        return self.decode(z), mu, logvar


model = CelebAVAE(latent_dim=128).to(device)
total_params = sum(p.numel() for p in model.parameters())
print(f'Parameters: {total_params:,}')

In [ ]:
optimizer   = optim.Adam(model.parameters(), lr=1e-3)
num_epochs  = 30
total_steps = num_epochs * len(loader)
step        = 0

# Distinct from 'vae_trained.pt' (used by explore_vae.ipynb / testing_shapes.ipynb for a
# different model). Save into /kaggle/working explicitly so the path is unambiguous.
SAVE_DIR  = '/kaggle/working' if os.path.exists('/kaggle/working') else '.'
SAVE_PATH = os.path.join(SAVE_DIR, 'celeba_vae_trained.pt')

for epoch in range(num_epochs):
    model.train()
    total_loss  = 0
    total_recon = 0
    total_kl    = 0
    for batch in loader:
        x = batch.to(device)
        # KL annealing: ramp from 0 to 1 over first half of training
        kl_weight = min(1.0, step / (total_steps * 0.5))
        step += 1

        optimizer.zero_grad()
        recon, mu, logvar = model(x)
        recon_loss = nn.functional.mse_loss(recon, x, reduction='sum')
        kl         = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())
        loss       = recon_loss + kl_weight * kl
        loss.backward()
        optimizer.step()
        total_loss  += loss.item()
        total_recon += recon_loss.item()
        total_kl    += kl.item()

    avg       = total_loss / len(dataset)
    avg_recon = total_recon / len(dataset)
    avg_kl    = total_kl / len(dataset)
    print(f'Epoch {epoch+1:02d}/{num_epochs}  loss: {avg:.2f}  recon: {avg_recon:.2f}  '
          f'kl: {avg_kl:.2f}  kl_weight: {kl_weight:.3f}')

    # Checkpoint every 5 epochs (and on the last one) so a Kaggle session
    # disconnect doesn't lose all progress — there's always a recent file to download.
    if (epoch + 1) % 5 == 0 or epoch + 1 == num_epochs:
        torch.save({
            'model_state': model.state_dict(),
            'optimizer_state': optimizer.state_dict(),
            'epoch': epoch + 1,
            'latent_dim': 128,
        }, SAVE_PATH)
        print(f'Checkpoint saved to {os.path.abspath(SAVE_PATH)}')

print('Done. On Kaggle: download the checkpoint from the Output file browser now, '
      'or click "Save Version -> Save & Run All" to persist /kaggle/working permanently.')

In [ ]:
# Originals vs reconstructions
model.eval()
batch = torch.stack([dataset[i] for i in range(8)]).to(device)
with torch.no_grad():
    recon, _, _ = model(batch)

fig, axes = plt.subplots(2, 8, figsize=(16, 4))
for i in range(8):
    axes[0, i].imshow(batch[i].cpu().permute(1, 2, 0))
    axes[0, i].axis('off')
    axes[1, i].imshow(recon[i].cpu().permute(1, 2, 0))
    axes[1, i].axis('off')
axes[0, 0].set_title('Original', loc='left', fontsize=12)
axes[1, 0].set_title('Reconstructed', loc='left', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# Random samples from prior
model.eval()
with torch.no_grad():
    z = torch.randn(16, 128).to(device)
    samples = model.decode(z)

fig, axes = plt.subplots(2, 8, figsize=(16, 4))
for i, ax in enumerate(axes.flatten()):
    ax.imshow(samples[i].cpu().permute(1, 2, 0))
    ax.axis('off')
plt.suptitle('Random samples from latent space')
plt.tight_layout()
plt.show()

In [ ]:
# Latent space interpolation between two faces
model.eval()
x1 = dataset[0].unsqueeze(0).to(device)
x2 = dataset[1000].unsqueeze(0).to(device)

with torch.no_grad():
    mu1, _ = model.encode(x1)
    mu2, _ = model.encode(x2)

n_steps = 10
alphas  = torch.linspace(0, 1, n_steps)
frames  = []
with torch.no_grad():
    for alpha in alphas:
        z   = alpha * mu1 + (1 - alpha) * mu2
        img = model.decode(z).squeeze(0).cpu().permute(1, 2, 0)
        frames.append(img)

fig, axes = plt.subplots(1, n_steps + 2, figsize=(22, 2.5))
axes[0].imshow(x1.squeeze().cpu().permute(1, 2, 0))
axes[0].set_title('A')
axes[0].axis('off')
for i, img in enumerate(frames):
    axes[i + 1].imshow(img)
    axes[i + 1].set_title(f'{alphas[i]:.1f}')
    axes[i + 1].axis('off')
axes[-1].imshow(x2.squeeze().cpu().permute(1, 2, 0))
axes[-1].set_title('B')
axes[-1].axis('off')
plt.suptitle('Latent interpolation: face A → face B')
plt.tight_layout()
plt.show()